In [9]:
import requests
import re
import pandas as pd
from datetime import datetime, date, timedelta
from bs4 import BeautifulSoup
from time import sleep
from random import random, randint
from tqdm.notebook import tqdm
tqdm.pandas()

/Users/gurumanikanta/anaconda3/lib/python3.7/site-packages/tqdm/std.py:651: FutureWarning: The Panel class is removed from pandas. Accessing it from the top-level namespace will also be removed in the next version
  from pandas import Panel


In [91]:
start_of_time = date(1900,1,1)
req_date = date(2019,1,1)
#(req_date - start_of_time) calculates days between the given dates i.e, excluding req_date and start_of_time
#add 2 to compensate for them
days = (req_date - start_of_time).days + 2
article_date = list()
article_link = list()
article_heading = list()
while req_date < datetime.today().date():
    year, month = str(req_date.year), str(req_date.month)
    base = "https://timesofindia.indiatimes.com/"
    date =  year + '/' + month + '/' + str(req_date.day)
    location = '/archivelist/'
    y1, m1 = 'year-' + year, ',month-' + month
    starttime = ',starttime-' + str(days) + '.cms'
    #sample url
    #"https://timesofindia.indiatimes.com/2019/1/1/archivelist/year-2019,month-1,starttime-43466.cms"
    url = base + date + location + y1 + m1 + starttime

    response = requests.get(url)
    soup = BeautifulSoup(response.content,'html.parser')
    mytable = soup.find_all('table')[1]
    for td in mytable.findChildren('tr')[2].findChildren('td'):
        for link in td.findChildren('a'):
            article_link.append(link.get('href'))
            article_heading.append(link.get_text())
            article_date.append(req_date)
        
    req_date = req_date + timedelta(days=1)
    days += 1

    time.sleep(random()*randint(4,19))


In [93]:
toi_data = DataFrame(columns=['date','heading','link'])
toi_data['date'] = article_date
toi_data['heading'] = article_heading
toi_data['link'] = article_link

In [81]:
from tqdm.notebook import tqdm
def find_category(link):
    if "http://timesofindia.indiatimes.com" not in link:
            link = "http://timesofindia.indiatimes.com/" + link
    try: 
        return re.findall(".*//(.*?)/.*",link)[0] 
    except Exception as e:
        return None

tqdm.pandas()
toi_data['category'] = toi_data['link'].progress_apply(find_category)

#pd.value_counts(toi_data['category'])
#3 categories like a pocket notes for top news have been left out, also the number of articles is very less

cat_div_class = {'Normal' : ['entertainment','life-style','citizen-reporter','business','sports','tv','home',
                             'politics','gadgets-news','astrology','auto','most-searched-products','gaming',
                             'companies','nri','best-products','independence-day-2018','people','trend-tracking',
                            'spotlight','venture-capital','viral-news','web-series','times-fact-check',
                             'toi-homepage-slug','brandwire','global','toi-features'],
                 '_3WlLe clearfix' : ['city','india','elections','world']}

'def find_category(link):\n    if "http://timesofindia.indiatimes.com" not in link:\n            link = "http://timesofindia.indiatimes.com/" + link\n    try: \n        return re.findall(".*//(.*?)/.*",link)[0] \n    except Exception as e:\n        return None\n\ntqdm.pandas()\ntoi_data[\'category\'] = toi_data[\'link\'].progress_apply(find_category)\n\n#pd.value_counts(toi_data[\'category\'])\n#3 categories like a pocket notes for top news have been left out, also the number of articles is very less\n\ncat_div_class = {\'Normal\' : [\'entertainment\',\'life-style\',\'citizen-reporter\',\'business\',\'sports\',\'tv\',\'home\',\n                             \'politics\',\'gadgets-news\',\'astrology\',\'auto\',\'most-searched-products\',\'gaming\',\n                             \'companies\',\'nri\',\'best-products\',\'independence-day-2018\',\'people\',\'trend-tracking\',\n                            \'spotlight\',\'venture-capital\',\'viral-news\',\'web-series\',\'times-fact-check\',\n

In [48]:
toi_data.to_csv('/Users/gurumanikanta/Documents/DS-Analytics/SupervisedLearning/TPI/TOI/toi_data.csv')
#toi_data = pd.read_csv('/Users/gurumanikanta/Documents/DS-Analytics/SupervisedLearning/TPI/TOI/toi_data.csv')

In [ ]:
#loading the previous file with link,heading,date and category
toi_data = pd.read_csv('/Users/gurumanikanta/Documents/DS-Analytics/SupervisedLearning/TPI/TPI/TOI/toi_data.csv')
toi_data.columns

In [16]:
def get_article_content(row):
    category = row['category']
    try:
        response = requests.get(row['link'])
        soup = BeautifulSoup(response.content,'html.parser')
        sleep(random()*randint(1,8))
    except Exception as e:
        print('Error at index: {} \nmessage: {}'.format(row.name,e))
        return None
    
    if (category in cat_div_class['Normal']) and (response.status_code == 200):
        try:
            return soup.find_all('div',attrs={'class':'Normal'})[0].get_text()
        except:
            return None
    elif (category in cat_div_class['_3WlLe clearfix']) and (response.status_code == 200):
        try:
            return soup.find_all('div',attrs={'class':'_3WlLe clearfix'})[0].get_text()
        except:
            return None
    else:
        return None
        
    
toi_data['content'][13001:15000] = toi_data.iloc[13001:15000,:].progress_apply(get_article_content, axis=1, result_type='expand')


In [27]:
#toi_data['content'] = toi_data['content'].str.encode('utf-8',errors='surrogatepass')
toi_data[:13000].to_csv('/Users/gurumanikanta/Documents/DS-Analytics/SupervisedLearning/TPI/TPI/TOI/toi_15k.csv',encoding='utf8',index=False)


In [30]:
toi_data.loc[14999,:]

date                                               2019-01-31
heading     CBSE 10th Board Exam 2019: Tips to score full ...
link        http://timesofindia.indiatimes.com//home/educa...
category                                                 home
content                                                   NaN
Name: 14999, dtype: object